#  Pruebas con Pytest

Existen varios marcos de pruebas en Python. Aquí trataremos con Pytest.

--- 
#  Pytest 

`Pytest` [(https://docs.pytest.org/)](https://docs.pytest.org/) permite escribir conjuntos de pruebas sencillos a la vez que admite diferentes tipos de pruebas, entre ellas las pruebas unitarias.

Para usar `pytest` es necesario instalar las librerías (en `uv` se instala como dependencia de desarrollo).

```python
    uv add --dev pytest coverage pytest-cov pytest-datafiles
```

## Construcción de los tests

* La mejor práctica es __crear una carpeta (por ejemplo `tests`)__ donde se incluyan todos los tests del paquete de usuario, aunque también se pueden escribir los test de prueba dentro del módulo (`archivo.py`) que finalmente irá a producción. 

* Una carpeta con todos los tests permite mantener el código de pruebas y el código de la aplicación separados.
Los archivos que contienen los test deben empezar por `test_`

* Carpeta de los tests. Todos los tests están en ficheros que se llaman `test_*.py` o `*_test.py`, mientras que los tests son funciones dentro de esos ficheros que se llaman `test_XXXX`.

```python
        tests
          |__ test_modulo1.py
          |__ test_modulo2.py
```

* Para ejecutar los tests de pruebas, Pytest buscará en directorios (y de forma recursiva) ficheros cuyo nombre  encaje con `test_*.py` o `*_test.py`.

* Pytest, requiere que se defina una función con `test_` antepuesto y que utilice las condiciones `assert` dentro de ellas.  Los `assert` permiten comprobar una condición. Si no se cumple, el test falla.



__Importante__: El directorio de `tests` no se distribuye con el paquete.


Todos los ejemplos que aparecen a continuación se encuentran en el proyecto: `funciones_pk`.

###  Ejemplo 1

Supongamos definida la función `gcd(a, b)`para calcular el máximo común divisor:

In [ ]:
import doctest
import pytest

def gcd(a: int, b: int) -> int:
    '''
    Returns the Greatest common divisor (gcd) of a and b.

    :param a: non negative integer
    :param b: positve integer
    :returns: the gcd of a and b
    :raises ValueError: if
      - a is negatve
      - b is non positive

    Examples
    --------
    >>> gcd(24, 16)
    8
    >>> gcd(30, 35)
    5
    >>> gcd(-3, 4)
    Traceback (most recent call last):
        ...
    ValueError: a = -3 must be non negative
    >>> gcd(3, 0)
    Traceback (most recent call last):
        ...
    ValueError: b = 0 must be a positive integer
    '''

    if a<0:
        raise ValueError(f'a = {a} must be non negative')
    if b<=0:
        raise ValueError(f'b = {b} must be a positive integer')

    while b!=0:
        r = a%b
        a = b
        b = r
    return a


Definimos varios tests para la función anterior:
* El primero `test_gcd_1()` símplemente comprueba que se cumplen los ejemplos de test que se encuentran en el docstring.
* El segundo `test_gcd_2()` , igualmente sin argumentos de entrada, comprueba con  `assert` que se verifiquen diferentes resultados de la función `gcd`.

```python
    # test_gcd.py

    # test definidos en el docstring
    ###################################
    def test_gcd_1():
        # Podemos ejecutar los tests en el docstring
        doctest.run_docstring_examples(gcd, globals())

    # Tests en el marco pytest
    ###############################
    def test_gcd_2():
        # test sin parámetros
        assert gcd(4, 6) == 2
        assert gcd(4, 7) == 1
        assert gcd(6, 7) == 1
        assert gcd(15, 5) == 5

```


Parametros:
* `gcd`: función objeto del test
* ` globals()`: es el contexto de ejecución (el diccionario de variables globales actuales). Proporciona a doctest el acceso a las funciones, variables e importaciones que los ejemplos del docstring puedan necesitar para ejecutarse correctamente.

## Ejecución de los tests

Para ejecutar las pruebas anteriores tenemos varias opciones:

### (1) Ejecución desde el propio modulo que contiene los test

Incluir el siguiente código dentro del módulo `.py` que contiene los tests:
  ```python

    # test_gcd.py
    import pytest    
    if __name__ == '__main__':
        # Ejecuta pytest en el directorio `tests/` para correr todos los archivos de prueba
        pytest.main(["__file__"])

  ```
### (2) Nuevo módulo que incluya código que ejecute los test

Crear un nuevo módulo llamado `run_pruebas.py` que incluya el siguiente código:
   ```python
   
    # run_pruebas.py
    import pytest    
    if __name__ == '__main__':
        # Ejecuta pytest en el directorio `tests/` para correr todos los archivos de prueba
        pytest.main(["../tests/"])

   
   ```

   Se ha de escribir bien la ruta a la carpeta contenedora de los tests.

### (3) Desde la terminal:

| Objetivo de Ejecución | Comando con `uv` | Notas / Descripción |
| :--- | :--- | :--- |
| **Pytest** | `uv run python -m pytest` | Corre la suite de pruebas estándar de `pytest`  |
| **Pytest + Doctests** | `uv run python -m pytest --doctest-modules` | **Recomendado:** Busca tanto pruebas en `test_*.py` como en *docstrings* |
| **Detalle de los tests ejecutados** | `pytest -v` |  |
| **Pytest + Doctest + Reporte HTML** | `uv run python -m pytest --doctest-modules --cov=. --cov-report=html` | Ejecuta todo el conjunto de pruebas (`pytest` + `doctest`) y exporta el informe de cobertura directamente a HTML. |

---
## Definir test con parámetros

###  Parametrizar Pytest con el decorador  `@pytest.mark.parametrize`

Pytest permite el uso del decorador `@pytest.mark.parametrize` en las funciones de test. Este decorador permite parametrizar los argumentos de una función de prueba. Esto simplifica el probar una función con múltiples conjuntos de datos, ya que  evita tener que escribir pruebas repetitivas. Simplemente definimos los diferentes valores que queremos probar y Pytest se encarga de ejecutar la función de prueba con cada conjunto de datos.


####  Ejemplo 2:

El test `test_gcd_2()` definido anteriormente se puede reescribir usando el decorador:

```python
        # test_gcd.py

        # Tests en el marco pytest con decorador 
        #########################################

        valores  = [
        (4, 6, 2),
        (4, 7, 1),
        (6, 7, 1),
        (15, 5, 5)
        ]
        # Usamos el decorador parametrize para probar múltiples casos
        @pytest.mark.parametrize("a, b, resultado_esperado", valores)
        def test_gcd_22(a, b, resultado_esperado):
            # test sin parámetros
            assert gcd(a, b) == resultado_esperado
```


##  Manejo de excepciones con Pytest

Para verificar que se lanzan excepciones bajo ciertas condiciones, usaremos el contexto `pytest.raises()`. Esto  permite verificar que se lance una excepción específica durante la ejecución del código. 


###  Ejemplo 3

Comprobaremos que se lanza excepción de tipo `ValueError` en caso de que los valores de `a` o `b` sean `0`. También cuando algunos de los dos sea negativo.


```python
    @pytest.mark.parametrize(
            "a,b",[(0, 0), (-1, 2), (2, 0), (2, -11), (-1, 0)])
    def test_gcd_4(a,b):
        with pytest.raises(ValueError):
            gcd(a,b)

```

##  Fixtures de Pytest

A veces para realizar las pruebas se necesitan objetos más complejos, como por ejemplo, listas, diccionarios, arrays, dataframes, fechas, etc. 

__Permiten preparar datos o contexto reutilizable.__

La  función de `fixtures` en Pytest permite incorporar datos complejos en las pruebas. Básicamente, son funciones que se ejecutan antes de cada prueba para crear objetos que se usarán en pruebas posteriores. Por ejemplo, si tenemos varias pruebas que utilizan los mismos datos, podemos usar un fixture para manejar esos datos repetidos con una sola función.

### Ejemplo 4

Supongamos que queremos validar la función factorial que calcula el factorial de una lista de números.


```python

# funciones_utilities.py

def factorial(lista:list[int])-> list[int]:
    """
    Calcula el factorial de una lista de números entero no negativo. El cálculo se hará
    de forma iterativa o recursiva dependiendo del valor del número.

    El factorial de un número ``n`` se define como el producto de todos los
    enteros positivos desde 1 hasta ``n``. Por convención, el factorial de 0 es 1.

    :param lista: lista de números enteros no negativos.
    :type : lista
    :return: lista del factorial de n números dados.
    :rtype: lista de enteros
    :raises ValueError: Si alguno de los números proporcionados es negativo.

    Examples
    --------
    >>> r = factorial([1100])
    >>> str(r[0])[0:26] == '53437084880926377034242155'
    True
    >>> factorial([0])
    [1]
    >>> factorial([1])
    [1]
    >>> factorial([-4])
    Traceback (most recent call last):
      ...
    ValueError: El número debe ser un entero no negativo.
    """
    res = []
    for n in lista:
        if n > 1000:
            res.append(factorial_iterativo(n))
        else:
            res.append(factorial_rec(n))
    return res
```

El test sería:

```python

    # test_funciones.py

    # Pytest con fixtures
    # Definimos un fixture llamado 'datos_input'
    @pytest.fixture
    def datos_input():
        return [1, 2, 3, 4, 5]

    @pytest.fixture
    def datos_output():
        return [1, 2, 6, 24, 120]

    # Usamos el fixture en una prueba de factorial
    def test_factorial_4(datos_input, datos_output):
        assert factorial(datos_input) == datos_output
```

El fixture `datos_input` devuelve una lista de números que se puede usar en las pruebas. Del mismo modo, el fixture `datos_output` devuelve otra lista del mismo tamaño con los resultados. 
La función de prueba `test_factorial_4` utiliza ambos fixtures  para verificar la función factorial.

##  Ficheros

Es posible  utilizar un fixture personalizada en pytest para cargar archivos de datos desde una carpeta especificada (datafiles). La marca `pytest.mark.datafiles` no es una funcionalidad estándar de pytest pero se  proporciona con el paquete `pytest-datafiles`.

##  Ejemplo

Supongamos que queremos probar la siguiente función:

In [ ]:
# ingesta_datos_api.py

import pandas as pd
import importlib.resources

def read_data(filename: str|None = None) -> pd.DataFrame:
    """
    This function reads a file with the required format. It returns store dataframe
    >>> read_data().columns
    Index(['Codigo', 'Nombre', 'Precio'], dtype='object')
    """
    if filename == None:
        # importlib.resources.files('funciones_pk') devuelve la raiz del paquete
        filename = importlib.resources.files('funciones_pk').joinpath('data/store2.txt')

    df = pd.read_csv(filename, sep = '|')

    return df

El módulo `importlib.resources` (incluido en la librería estándar de Python) sirve para acceder a archivos de datos (como imágenes, archivos JSON, bases de datos SQLite, modelos, plantillas o archivos .txt) de manera segura.

Para leer un archivo de datos dentro de un proyecto se usaban rutas de archivo relativas con el módulo `os` o `pathlib`, pero esto falla:
* en paquetes empaquetados o comprimidos(wheel) donde las rutas de sistema de archivos tradicionales no existen en el disco y la lectura falla.
* cuando hay diferencias entre Sistemas Operativos: Las separaciones de rutas (/ vs \) requieren un manejo cuidadoso en Windows y Linux.

Nos crearemos el siguiente código:


```python
    from pathlib import Path

    # Definición del directorio de datos
    # FIXTURE_DIR define la carpeta donde se encuentran los archivos de prueba, 
    # relativa al archivo actual
    FIXTURE_DIR = Path(__file__).parent.resolve() / 'data'

    # Posteriormente, pytest.mark.datafiles marca los archivos que queremos 
    # incluir como datos de prueba.
    # se crea un fixture datafiles donde se copian los ficheros indicados que usarán los tests
    FILES = pytest.mark.datafiles(
        FIXTURE_DIR / 'store.txt',
        FIXTURE_DIR / 'store2.txt',
    )

    @pytest.fixture
    def files(datafiles):
        # devuelve un diccionario del tipo:
        #  {'nombre_archivo': 'ruta/completa/temporal'}
        d = dict()
        # Recorre cada archivo presente en la carpeta temporal.
        for fname in datafiles.iterdir():   
            d[fname.name] = fname.as_posix()
        return d
```

* En el código anterior, la constantes `FIXTURE_DIR` define la carpeta donde se encuentran los archivos de prueba, relativa al archivo actual.


```python
        tests/
           test_funciones.py
           data/
               store.txt
               store2.txt
               empleados.csv
```


* Posteriormente, `pytest.mark.datafiles` marca los archivos que queremos incluir como datos de prueba.
Antes de ejecutar cada test decorado con `@FILES`, se  copian esos archivos a un directorio temporal y los hace accesibles mediante la fixture `datafiles`, es decir,  no usa directamente la carpeta original, sino una copia aislada. Esto evita 
modificar archivos reales y  conflictos entre tests.

* `datafiles` es una fixture automática del plugin que contiene el directorio temporal con los archivos copiados.


* La fixture `files` itera sobre los archivos proporcionados por `datafiles`,  devolviendo un diccionario donde las claves son los nombres de archivo y los valores son las rutas completas.

```python
{
  'store.txt': '/ruta/temp/store.txt',
  'store2.txt': '/ruta/temp/store2.txt',
  'empleados.csv': '/ruta/temp/empleados.csv'
}
```

* Con la fixture `files` creamos dos nuevas fixtures: `dataset1` y `dataset2` que devuelven dos dataframes con los datos de los ficheros. 

```python

    @pytest.fixture
    def dataset_empleados(files):
        return pd.read_csv(files['empleados.csv'], sep = ',')


    @pytest.fixture
    def dataset_store(files):
        return pd.read_csv(files['store.txt'], sep = '|')
        
```

Ahora, para testear la función `read_data` usamos el decorador `@FILES`.
Cada tests que queramos que usen los ficheros los debemos
marcar con `@FILES`.

```python
    # Test 2: test read_data. 
    # Comprobar pasando un archivo explícito mediante la fixture 'files'
    @FILES
    def test_read_data_with_custom_file(files):
        # 'files' es el diccionario que mapea 'store2.txt' a su ruta temporal
        path_store2 = files['store2.txt']
        df = read_data(path_store2)              # test de read_data
        assert isinstance(df, pd.DataFrame)
        assert len(df) > 0
        assert 'Codigo' in df.columns

    # Test 3: 
    @FILES
    def test_mayor(dataset_empleados):
            assert type(dataset_empleados) == pd.DataFrame
            assert mayor(dataset_empleados) == 35
```

En este caso, cuando se ejecuta el test, Pytest ve `@FILES`, copia los archivos al directorio temporal, ejecuta las fixtures y se inyecta `dataset_empleados`. Cada vez que se ejecuta el test se hace una copia de esos ficheros que al finalizar se borran.

## Objetos simulados (mocking)

Usamos objetos simulados cuando necesitamos hacer testing de funciones que usan objetos complejos (bases de datos, accesos a red). Debemos hacer objetos que simulen esos objetos complejos o no controlables.


Tenemos un módulo `tomorrow.py`

In [ ]:
# tomorrow.py

from datetime import date , timedelta

def calcular_tomorrow() -> date :
    return date.today() + timedelta( days =1)

El problema para hacer test de `calcular_tomorrow` es que  `date.today()`  depende del día real en que ejecutes el test. Una solución es simular la fecha actual.

Se crea una clase que hereda de date, pero sobrescribiendo el método `today()`. En vez de devolver la fecha real, devuelve el valor de la variable today que llega desde parametrize.

```python
    # test_tomorrow.py

    import tomorrow
    import pytest
    import datetime
    from datetime import date

    @pytest.mark.parametrize("today , expected ", [
            ( date(2021 , 1, 1), date(2021 , 1, 2)) ,
            ( date(2021 , 12, 31) , date(2022 , 1, 1)) ,
            ( date(2020 , 2, 28) , date(2020 , 2, 29)) ,
            ( date(2020 , 2, 29) , date(2020 , 3, 1)) ,
            ( date(2021 , 2, 28) , date(2021 , 3, 1)) ,
            ])
    def test_tomorrow( monkeypatch , today , expected ):
        class MockDate( date ):
            @staticmethod
            def today():
                return today

        # Sustitución con monkeypatch
        monkeypatch.setattr( tomorrow , "date", MockDate )
        assert tomorrow.calcular_tomorrow() == expected
```

`monkeypatch` es la herramienta de pytest para cambiar temporamente valores. 

La instrucción:
```python
    monkeypatch.setattr(tomorrow, "date", MockDate)
```
hace que, dentro del módulo `tomorrow`, donde antes se usaba la clase `date`, ahora se usa `MockDate`. De esta forma, cuando `tomorrow.calcular_tomorrow()` llama a `date.today()`, en realidad está llamando a `MockDate.today()`, que devuelve la fecha simulada.

## Simulado request

los problemas que podemos tener son:
* La web puede fallar en el momento de los tests.
* Se tiene que descargar los datos de la red. Puede ser lento.
* No controlamos el contenido de la red

En el siguiente ejemplo mostramos un test de pytest que simula llamadas a Wikipedia para comprobar una función `traduce_wikipedia(palabra, idioma)` sin hacer peticiones reales a Internet. Combina fixtures, mocking y parametrización.

In [ ]:
# traductor_html.py

import requests
import bs4

def traduce_wikipedia(palabra: str, codigo_idioma: str) -> str | None:
    link = f"https://es.wikipedia.org/wiki/{palabra}"

    try:
        resp = requests.get(link,  timeout=15)
        if resp.status_code != 200:
            return None
    except requests.RequestException as e:
        print("Error de conexión:", e)
        return None

    page = bs4.BeautifulSoup(resp.text, features="html.parser")
    trad = page.find_all(
        "a",
        attrs={"class": "interlanguage-link-target", "hreflang": codigo_idioma},
        limit=1
    )
    if len(trad) > 0:
        resp_text = trad[0].attrs["data-title"]
        return resp_text.strip()

    return None


Creamos una fixture  que abre un archivo local `apartamento.html`. Este HTML es una página de Wikipedia guardada en local que usaremos (su contenido) como si fuera la respuesta real de la web. Esta fixture lee todo el contenido el fichero html y lo devuelve.

```python
    # test_traductor_html.py

    # Tests para funciones que trabajan con ficheros
    from pathlib import Path

    #  FIXTURE_DIR define la carpeta donde se encuentra el archivo de prueba,
    #  relativa al archivo actual. Esto permite localizar el fichero 
    #  apartamento.html sin depender de la carpeta actual de ejecución.
    FIXTURE_DIR = Path(__file__).parent.resolve() / 'data'

    #  Fixture para leer el HTML . SE abre el fichero HTML y devuelve su
    #  contenido como texto. Así el test no necesita leer el HTML manualmente.
    #  Esto es útil porque queremos simular la respuesta real de Wikipedia.
    @pytest.fixture
    def apartamento():
        path = FIXTURE_DIR / "apartamento.html"
        with open(path, encoding="utf-8") as f:
            return f.read()
```


La clase `MockResponse` simula el objeto que devuelve `requests.get()`. Esto permite probar el código sin conexión a Internet.

```python
    class MockResponse :
        def __init__ (self , text ):
            self.status_code = 200
            self.text = text

            
    @pytest.fixture
    def mock_resp_apartamento(apartamento):
        return MockResponse(apartamento)
```

La fixture `mock_resp_apartamento` encadena fixtures:

* Recibe el HTML leído por apartamento.
* Devuelve un objeto MockResponse que lo contiene.

el resultado es un objeto equivalente a una respuesta HTTP real.

El test se ejecuta una vez por cada tupla de traducciones.

In [14]:
traducciones = [
        ( 'apartamento' , 'de' , 'Wohnung' ),
        ( 'apartamento' , 'pt' ,  'Apartamento' ),
        ( 'apartamento' , 'en' , 'Apartment' ),
        ( 'apartamento' , 'fr' ,  'Appartement' ),
        ( 'apartamento' , 'XYZ' , None ),
        ('XYZ' , 'en' , None )
]

El test que creamos usa un Mock de `requests.get` que no es más que una función falsa que sustituye a `requests.get`. Si se busca "apartamento", devuelve el HTML real guardado localmente. Si se busca otra palabra,  devuelve una respuesta vacía.

```python
    import funciones_pk.traductor_html as traductor

    @pytest.mark.parametrize('palabra , idioma , esperado', traducciones)
    def test_traduce_wikipedia_2 (monkeypatch ,
                                mock_resp_apartamento ,
                                palabra , idioma , esperado ):
        def mock_get(*args , **kwargs ):
            if palabra == 'apartamento' :
                return mock_resp_apartamento
            else :
                return MockResponse('')
                
        # Sustitución con monkeypatch        
        monkeypatch.setattr(traductor.requests , 'get' , mock_get )
        assert traductor.traduce_wikipedia( palabra , idioma ) == esperado
```

`monkeypatch` es la herramienta de pytest para cambiar temporamente valores. Esto hace que, durante el test `requests.get(...)` no se  llame a Internet, sino a `mock_get(...)`.